# ADR Lab 1. Generowanie danych

In [1]:
print('Hello')

Hello


Komnurka w notatniku to też terminal, dodajemy ! żeby pokazać (dla dnas), że to komenda z termianla

In [2]:
!ls

Untitled.ipynb	Untitled1.ipynb  nul


In [4]:
!ls -all

total 16
drwxrwxrwx 1 root   root  4096 Mar 30 10:30 .
drwsrws--- 1 jovyan users 4096 Mar 30 10:08 ..
drwxrwxrwx 1 root   root  4096 Mar 30 10:28 .ipynb_checkpoints
-rwxrwxrwx 1 root   root     0 Mar 27 12:40 .notremove
-rwxrwxrwx 1 root   root  1102 Mar 27 13:04 Untitled.ipynb
-rw-r--r-- 1 jovyan users 1742 Mar 30 10:30 Untitled1.ipynb
-rw-r--r-- 1 jovyan users  247 Mar 27 12:49 nul


In [5]:
!pwd

/home/jovyan/notebooks


## Zad. 2
Tworzymy producenta

In [7]:
%%file producer.py  
# Jezeli uruchome to kod zostanie zapisany do pliku producer.py, uruchomienie może nadpisać istniejący plik
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',                             # server
    value_serializer=lambda v: json.dumps(v).encode('utf-8')     # Serializator, wyrzucamy dane
)
# Zwróc uwage na to że wstawiamy funkcje lambda jako parametr
# jest to możliwe dzięki temu że wyszystko jest w Pythonie obiektem

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    # producer.send('lab4', value=tx) # do jakiego pliku, jaka wartość ma zostać wysłana
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


## Zad 3.1
Tworzymy konsumenta z lab4/transactions

In [16]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) # Teraz ładujemy dane
)

for message in consumer:
    if message.value['amount'] > 3000:
        print(f"ALERT: {message.value}")
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź amount > 1000, jeśli tak — wypisz ALERT

Overwriting consumer_filter.py


## Zad. 3.2
Tworzymy innego konsumenta który dodaje pole risk level: risk_level: amount > 3000 → “HIGH” - amount > 1000 → “MEDIUM” - pozostałe → “LOW” <br>
konsument ten będzie "odbiornikiem" który nasłuchuje nowe transakcje z kafki i dla każdej opisuje poziom ryzyka. <br>
Jest to przetwarzanie __bezstansowe__ - każda wiadomość była __analizowana osobno__

In [8]:
%%file consumer_enrich.py
from kafka import KafkaConsumer # Konsument łączy się z kafką i odbiera wiadomości
import json                     # Zamiana bajtów na pythonowy dict

# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

consumer = KafkaConsumer(
    'transactions',                       # Czytanie z tematu utw. w części 1
    bootstrap_servers='broker:9092',      # Kafka działa pod nazwą `broker` w Dockerze na procie `9092`
    group_id='consumer-enrich-group-v2',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))   # bajty -> tekst -> słownik                      
)

print("Nasłuchuje i dodaje risk_level...")
for message in consumer:
    event = message.value # dane wiadomości (message - obiekt wiadomości, treść wiadomości JSON message.value
    amount = event['amount']
    if amount > 3000:
        risk_level = "HIGH"
    elif amount > 1000:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"
    event["risk_level"] = risk_level # Dodanie pola ryzyka do słownika wiadomości
    print(event)
                                            

Overwriting consumer_enrich.py


Trzeba użyć __innego group_id__, czyli __inną grupę__ dla `consumer_enrich.py` niż `consumer.py` ponieważ
- jeżeli 2 różne programy konsumenckie mają tę samą grupę, to Kafka dzieli wiadomości __między nich__
- jeżeli mają różne grupy, to każdy dostaje swój własny pełny strumień
Chcemy aby `consumer_enrich.py` działał niezależnie od innych konsumentów, więc przypisujemy mu własną grupę

## Zad 4.1
Zliczanie wiadomości per sklep. <br>
Ten konsument będzie "pamiętał" poprzednie wiadomości, czyli __stan__, jest to przetwarzanie stanowe.

In [10]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'consumer-count_group',
    auto_offset_reset = 'earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Zmienne stanu
store_counts = Counter() # Liczba transakcji dla każdego sklepu
total_amount = {}        # Słownik sum wartości transakcji dla każdego sklepu
msg_count = 0            # Licznik odebranych wiadomości

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. Zwiększ store_counts[store]
#   2. Dodaj amount do total_amount[store]
#   3. Co 10 wiadomości wypisz tabelę:
#      Sklep | Liczba | Suma | Średnia

print("Nasłuchuję i zliczam transakcje per sklep...")

# Wyciąganie danych z wiadomości
for message in consumer:
    event = message.value     # słownik transakcji
    store = event["store"]    # nazwa sklepu
    amount = event["amount"]  # kwota transakcji

    # 1. Zwiększ store_counts[store]
    store_counts[store] += 1

    # 2. Dodaj amount do total_amount[store]
    if store not in total_amount:
        total_amount[store] = 0.0
    total_amount[store] += amount

    msg_count += 1

    # 3. Co 10 wiadomości wypisz tabelę
    if msg_count % 10 == 0:
        print("\n" + "=" * 50)
        print(f"Podsumowanie po {msg_count} wiadomościach")
        print("=" * 50)
        print(f"{'Sklep':<12} {'Liczba':<8} {'Suma':<12} {'Średnia':<12}")

        for store_name in store_counts:
            count = store_counts[store_name]
            total = total_amount[store_name]
            avg = total / count
            print(f"{store_name:<12} {count:<8} {total:<12.2f} {avg:<12.2f}")

Overwriting consumer_count.py


## Zad 4.2
Statystyki per kategoria

In [14]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

# TWÓJ KOD

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers = 'broker:9092',
    group_id = 'consumer-stats-group',
    auto_offset_reset = 'earliest',
    value_deserializer = lambda x: json.loads(x.decode('utf-8'))
)

stats = defaultdict(lambda: {
    "count": 0,
    "sum" : 0.0,
    "min": float("inf"),
    "max": float("-inf")
})

msg_count = 0

print("Nasłuchuję i liczę statystyki per kategoria...")

for message in consumer:
    event = message.value
    category = event['category']
    amount = event['amount']

    stats[category]['count'] += 1
    stats[category]['sum'] += amount
    stats[category]['min'] = min(stats[category]['min'], amount)
    stats[category]['max'] = max(stats[category]['max'], amount)

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n" + "=" * 65)
        print(f"Statystyki po {msg_count} wiadomościach")
        print("=" * 65)
        print(f"{'Kategoria':<15} {'Liczba':<8} {'Przychód':<12} {'Min':<12} {'Max':<12}")

        for cat in sorted(stats):
            count = stats[cat]['count']
            total = stats[cat]['sum']
            min_amount = stats[cat]['min']
            max_amount = stats[cat]['max']
    
            print(f"{cat:<15} {count:<8} {total:<12.2f} {min_amount:<12.2f} {max_amount:<12.2f}")

Overwriting consumer_stats.py


## Zad 5.2
1. __Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?__ <br>
   consumer_filter.py uruchomi się poprawnie, ale będzie czekał na wiadomości których nie ma (czyta wiadmości z tematu transactions, do którego producer.py powinien wysyłać wiadmości, ale tego nie robi)
2. __Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?__<br>
   Kafka potraktuje ich jako jedną grupę konsumentów i będzie rozdzielać wiadmości między nich, zamiast wysyłać pełen strumień do bou
3. __Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?__ <br>
   Przetwarzanie bezstanwoe analizuje każde zdarzenie niezależnie, bez pamięci o poprzednich, a przetwarzanie stanowe przechowuje stan między kolejnymi wiadomościami, np. licznik, sumy, minima i maksima

## Praca domowa
Konsument wykrywający anomalię prędkości transakcji

In [19]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
from collections import defaultdict, deque
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-anomaly-group',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Dla każdego usera trzymamy czasy jego ostatnich transakcji
user_transactions = defaultdict(deque) # klucz 

print("Nasłuchuję i wykrywam anomalie prędkości transakcji...")

for message in consumer:
    event = message.value
    user_id = event["user_id"]
    tx_id = event["tx_id"]
    amount = event["amount"]
    store = event["store"]
    category = event["category"]
    timestamp_str = event["timestamp"]

    # Zamiana tekstu ISO na datetime
    event_time = datetime.fromisoformat(timestamp_str)

    # Pobranie kolejki czasów dla danego usera
    tx_times = user_transactions[user_id]

    # Dodanie nową transakcję
    tx_times.append(event_time)

    # Usuńnięcie transakcje starsze niż 60 sekund
    window_start = event_time - timedelta(seconds=60)
    while tx_times and tx_times[0] < window_start:
        tx_times.popleft()

    # Jeśli w ostatnich 60 sekundach są więcej niż 3 transakcje -> alert
    if len(tx_times) > 3:
        print(
            f"ALERT: user {user_id} wykonał {len(tx_times)} transakcji "
            f"w ciągu 60 sekund | ostatnia: {tx_id} | {amount:.2f} PLN | "
            f"{store} | {category} | {timestamp_str}"
        )

Writing consumer_anomaly.py


Alert przy wejściu w stan anomali:

In [ ]:
%%file consumer_anomaly2.py
from kafka import KafkaConsumer
from collections import defaultdict, deque
from datetime import datetime, timedelta
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='consumer-anomaly-group-v2',
    auto_offset_reset='earliest',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_transactions = defaultdict(deque)
user_alert_state = defaultdict(lambda: False)

print("Nasłuchuję i wykrywam anomalie prędkości transakcji...")

for message in consumer:
    event = message.value
    user_id = event["user_id"]
    tx_id = event["tx_id"]
    amount = event["amount"]
    timestamp_str = event["timestamp"]

    event_time = datetime.fromisoformat(timestamp_str)
    tx_times = user_transactions[user_id]
    tx_times.append(event_time)

    window_start = event_time - timedelta(seconds=60)
    while tx_times and tx_times[0] < window_start:
        tx_times.popleft()

    is_anomaly = len(tx_times) > 3

    if is_anomaly and not user_alert_state[user_id]:
        print(
            f"ALERT: user {user_id} wykonał {len(tx_times)} transakcji "
            f"w ciągu 60 sekund | ostatnia transakcja: {tx_id} | {amount:.2f} PLN"
        )

    user_alert_state[user_id] = is_anomaly